# Step 2 — Embed & Index
Creates a Cortex Search Service over CHUNKED_DOCUMENTS using Arctic Embed (managed automatically by Snowflake Cortex).

**Constraints:** All embedding/indexing happens inside Snowflake Cortex — no external APIs.

In [ ]:
import time
from snowflake.snowpark import Session

try:
    session = Session.builder.getOrCreate()
except:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()

TARGET_DB     = "RAG_HACKATHON_DB"
TARGET_SCHEMA = "RAG_SCHEMA"
TARGET_WH     = "SYSTEM$STREAMLIT_NOTEBOOK_WH"   # ✅ FIXED
SERVICE_NAME  = "ECONOMIC_SEARCH"

session.sql(f"USE DATABASE {TARGET_DB}").collect()
session.sql(f"USE SCHEMA {TARGET_SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {TARGET_WH}").collect()

print("Session ready")
print("DB:", session.get_current_database())
print("Schema:", session.get_current_schema())
print("WH:", session.get_current_warehouse())

In [ ]:
# STEP 2 — FIX: Create a proper warehouse for Cortex Search

REAL_WH = "RAG_SEARCH_WH"

session.sql(f"""
CREATE WAREHOUSE IF NOT EXISTS {REAL_WH}
WITH WAREHOUSE_SIZE = 'XSMALL'
AUTO_SUSPEND = 60
AUTO_RESUME = TRUE
""").collect()

print("Warehouse ready:", REAL_WH)

In [ ]:
print('Creating CORTEX SEARCH SERVICE ...')

session.sql(f"""
CREATE OR REPLACE CORTEX SEARCH SERVICE {TARGET_DB}.{TARGET_SCHEMA}.{SERVICE_NAME}
  ON chunk_text
  ATTRIBUTES source, doc_id, title, chunk_index
  WAREHOUSE = {REAL_WH}
  TARGET_LAG = '1 hour'
  AS (
    SELECT chunk_text, source, doc_id, title, chunk_index
    FROM {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS
  )
""").collect()

print('CREATE command submitted — monitoring status...')

In [ ]:
# DEBUG — inspect DESCRIBE output exactly
rows = session.sql(
    f"DESCRIBE CORTEX SEARCH SERVICE {TARGET_DB}.{TARGET_SCHEMA}.{SERVICE_NAME}"
).collect()

print("Returned rows:\n")
for r in rows:
    print(r)
    print(r.as_dict())
    print("-" * 80)

In [ ]:
# CELL 4 — Check Cortex Search Service status (FINAL)

rows = session.sql(
    f"DESCRIBE CORTEX SEARCH SERVICE {TARGET_DB}.{TARGET_SCHEMA}.{SERVICE_NAME}"
).collect()

info = rows[0].as_dict()

print("Cortex Search Service Status\n")
print("Service Name      :", info.get("name"))
print("Database          :", info.get("database_name"))
print("Schema            :", info.get("schema_name"))
print("Warehouse         :", info.get("warehouse"))
print("Search Column     :", info.get("search_column"))
print("Attributes        :", info.get("attribute_columns"))
print("Rows Indexed      :", info.get("source_data_num_rows"))
print("Embedding Model   :", info.get("embedding_model"))
print("Indexing State    :", info.get("indexing_state"))
print("Serving State     :", info.get("serving_state"))
print("Refresh Mode      :", info.get("refresh_mode"))
print("Created On        :", info.get("created_on"))
print("Data Timestamp    :", info.get("data_timestamp"))

if str(info.get("indexing_state", "")).upper() == "ACTIVE" and str(info.get("serving_state", "")).upper() == "ACTIVE":
    print("\nStep 2 COMPLETE — Cortex Search Service is ACTIVE and ready for Step 3.")
else:
    print("\nService is not fully active yet. Re-run this cell after a short wait.")

In [ ]:
# CELL 5 — Test retrieval with a sample query (final cleaner version)
from snowflake.core import Root

root = Root(session)
svc = root.databases[TARGET_DB].schemas[TARGET_SCHEMA].cortex_search_services[SERVICE_NAME]

test_queries = [
    "What are recent trends in banking loan performance?",
    "Describe population demographics and income distribution",
    "What economic indicators are most relevant to credit risk?"
]

for q in test_queries:
    resp = svc.search(query=q, columns=["chunk_text", "source", "title"], limit=3)
    results = resp.model_dump().get("results", [])

    print(f'\nQuery: "{q}"')
    for i, r in enumerate(results, 1):
        print(f'  [{i}] [{r["source"]}] {str(r.get("title",""))[:50]}')
        print(f'       {r["chunk_text"][:150]}...')

print("\nStep 2 COMPLETE — Cortex Search Service is live and returning results")